# 01 · 期货数据准备（唯一主版本）

在 project 根目录安装 `python -m pip install -e ".[dev]"`，选择同一个 Python kernel。默认复用已下载原始行情，不联网。数据定义采用 Table Internet-I 逐品种通知日惯例，不再使用月初下界或严格日期平行版本。范围取表内第一个日期，末端保留在元数据供审计；此前两端月收益比对结果相同。营业日使用美国联邦假日日历代理，尚非完整历史交易所日历。原 vendor 日期不改写，Barley 候选和不可得 Propane 不伪造收益。

In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if (ROOT / 'project' / 'pyproject.toml').exists(): ROOT = ROOT / 'project'
assert (ROOT / 'configs' / 'commodity_map.csv').exists(), 'Open the project folder first'
sys.path.insert(0, str(ROOT / 'src'))
from project.paths import ensure_directories
ensure_directories()
pd.set_option('display.max_columns', 14)


In [ ]:
from project.acquisition import connect_wrds,download_futures
RUN_DOWNLOAD=False
if RUN_DOWNLOAD:
    conn=connect_wrds()
    try: download_futures(conn)
    finally: conn.close()
from project.data import build_dataset
panel=build_dataset()
display(panel.groupby('paper_code').agg(n=('excess_return','count'),first=('date','min'),last=('date','max')))
display(panel.reason.value_counts().to_frame('months'))


月末按交割月份排序，从第二近起选通知日和截止界限晚于下月末的合约。两月末使用同一 futcode。若第二近不安全则向后选，这是实现规则，论文示例未明确覆盖该边界。缺到期日使用交割月末假设并标注；现金结算品种采用附录最后交易日惯例。不得把附录估算日期称为实际历史FND。

In [ ]:
from project.portfolios import make_portfolios
factors,weights=make_portfolios(panel)
factors.to_parquet(ROOT/'data/processed/portfolios.parquet')
weights.to_parquet(ROOT/'data/processed/weights.parquet',index=False)
display(factors[['AVG','C5','M5']].describe())


输出：commodity_monthly.parquet（完整历史）、commodity_monthly_2000_2025.parquet（Proposal）、portfolios.parquet、weights.parquet。组合规则仍保留少于每侧5个时缺失的主规则；不要将未匹配的CARRY宣称精确复现。